# Tabela do benchmark — treino por épocas × one-shot, por binarização e semente

A semente muda a divisão treino/validação/teste, o mapeamento das RAMs e o embaralhamento das épocas, então cada semente é uma repetição do experimento. Em **cada semente**, cada modelo fica com a configuração de **melhor acurácia de validação** e a tabela mostra o **teste** dela (o teste não participa da escolha). As sementes aparecem lado a lado, seguidas da **média ± desvio-padrão** entre elas (o desvio só existe com 2 ou mais sementes; mnist, fashion-mnist e jsc têm uma).

| coluna | o que é |
|---|---|
| n teste | tamanho do conjunto de teste (divisão estratificada 70/15/15, ou o teste oficial); é o mesmo em todas as sementes |
| sementes | quantas sementes o conjunto tem |
| Épocas sK | teste do treino por épocas escolhido na validação da semente K, entre bits × n-tupla × b × ρ₀ × τ (empates: correção mais fraca, depois menos bits e menor n-tupla — `summarize.choose`) |
| One-shot sK | teste do one-shot escolhido na validação da semente K, entre bits × n-tupla |
| Δ sK, z sK | Épocas − One-shot (pp) e o z de duas proporções Δ / √(p_E(1−p_E)/n + p_O(1−p_O)/n), conservador (ignora o pareamento); \|z\| ≥ 1,96 em negrito |
| média ± dp | média e desvio-padrão amostral (ddof = 1) entre as sementes |
| Δ > 0 | em quantas sementes o treino por épocas ganhou |

**Resumo** de cada binarização: com o **Δ médio de cada conjunto** (as sementes são agregadas antes, pois duas sementes do mesmo conjunto não são dois conjuntos independentes), vitórias/empates/derrotas, Δ médio e mediano, Wilcoxon (diferenças ≠ 0) e teste de sinais; e, por semente, vitórias/empates/derrotas e Δ médio.

Abaixo da tabela principal: validação e oráculo (maior teste do grid, só um teto) por semente, e a configuração escolhida em cada semente.

In [9]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display

# the notebook lives in experiments/, but also works when started from the repository root
HERE = Path.cwd() if (Path.cwd() / "summarize.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(HERE))
import datasets
import summarize as sm

RESULTS_DIR = HERE / "results"
DATA_DIR = HERE / "data"
SEEDS = None  # None: every seed with results for both models; or a list, e.g. [0, 1]
ENCODINGS = {"distributive": "Termômetro distributivo", "thermometer": "Termômetro linear"}

## Sementes disponíveis e tamanho do teste

Uma semente entra quando tem resultados dos dois modelos. O tamanho do teste segue a regra do `run_grid.split`: o teste oficial quando existe; senão, em cada classe com *k* exemplos, *k* − round(0,70·*k*) − round(0,15·*k*) (não depende da semente).

In [10]:
def test_size(name):
    ds = datasets.load(name, str(DATA_DIR))
    if ds.has_official_test:
        return len(ds.y_test)
    _, counts = np.unique(ds.y, return_counts=True)
    return int(sum(k - int(round(0.70 * k)) - int(round(0.15 * k)) for k in counts))


data = sm.load(str(RESULTS_DIR))
seeds_of = {}
for (name, encoding), seeds in sorted(data.items()):
    if encoding not in ENCODINGS:
        continue
    ok = [s for s, rows in sorted(seeds.items()) if (SEEDS is None or s in SEEDS) and len(sm.choose(rows)) == len(sm.MODELS)]
    skipped = [s for s in sorted(seeds) if s not in ok and (SEEDS is None or s in SEEDS)]
    if skipped:
        print("%s / %s: semente(s) %s fora (sem os dois modelos)" % (name, encoding, skipped))
    if ok:
        seeds_of[(name, encoding)] = ok

n_test = {name: test_size(name) for name in sorted({k[0] for k in seeds_of})}
pd.DataFrame([{"conjunto": k[0], "binarização": ENCODINGS[k[1]], "sementes": ", ".join(map(str, s)), "n teste": n_test[k[0]]}
              for k, s in sorted(seeds_of.items())])

jsc / thermometer: semente(s) [0] fora (sem os dois modelos)


,conjunto,binarização,sementes,n teste
0,australian,Termômetro distributivo,"0, 1",104
1,australian,Termômetro linear,"0, 1",104
2,blood,Termômetro distributivo,"0, 1",111
3,blood,Termômetro linear,"0, 1",111
4,christine,Termômetro distributivo,"0, 1",814
5,christine,Termômetro linear,"0, 1",814
6,fashion-mnist,Termômetro distributivo,1,10000
7,fashion-mnist,Termômetro linear,1,10000
8,higgs,Termômetro distributivo,"0, 1",14708
9,higgs,Termômetro linear,"0, 1",14708


## Um resultado por conjunto, binarização e semente

In [11]:
def seed_row(name, encoding, seed):
    rows = data[(name, encoding)][seed]
    chosen = sm.choose(rows)
    e_cfg, e_test, e_val = chosen["epochs + bleaching"]
    o_cfg, o_test, o_val = chosen["one-shot + bleaching"]
    pe, po, n = e_test["acc"], o_test["acc"], n_test[name]
    se = math.sqrt(pe * (1 - pe) / n + po * (1 - po) / n)
    return dict(
        dataset=name, encoding=encoding, seed=seed, n_test=n,
        epochs=100 * pe, oneshot=100 * po, delta=100 * (pe - po), delta_examples=int(round((pe - po) * n)),
        z=(pe - po) / se if se > 0 else 0.0,
        epochs_val=100 * e_val["acc"], oneshot_val=100 * o_val["acc"],
        oracle_epochs=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "epochs"),
        oracle_oneshot=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "oneshot"),
        epochs_bits=e_cfg["bits"], epochs_address=e_cfg["address"], epochs_b=e_cfg["b"],
        epochs_rho0=e_cfg["rho0"], epochs_tau=e_cfg["tau"], best_epoch=e_cfg["bestEpoch"], epochs_run=e_cfg["ran"],
        oneshot_bits=o_cfg["bits"], oneshot_address=o_cfg["address"])


long = pd.DataFrame([seed_row(name, encoding, seed) for (name, encoding), seeds in sorted(seeds_of.items()) for seed in seeds])

# mean and sample standard deviation over the seeds of each dataset and encoding (NaN with a single seed)
aggregate = long.groupby(["dataset", "encoding"]).agg(
    seeds=("seed", "count"), n_test=("n_test", "first"),
    epochs_mean=("epochs", "mean"), epochs_std=("epochs", "std"),
    oneshot_mean=("oneshot", "mean"), oneshot_std=("oneshot", "std"),
    delta_mean=("delta", "mean"), delta_std=("delta", "std"),
    wins=("delta", lambda d: int((d > 0).sum()))).reset_index()
aggregate

,dataset,encoding,seeds,n_test,epochs_mean,epochs_std,oneshot_mean,oneshot_std,delta_mean,delta_std,wins
0,australian,distributive,2,104,85.096154,2.039731,80.769231,1.359821,4.326923,0.679910,2
1,australian,thermometer,2,104,80.288462,0.679910,81.730769,2.719641,-1.442308,3.399552,1
2,blood,distributive,2,111,71.621622,0.637033,72.522523,1.911099,-0.900901,2.548133,1
3,blood,thermometer,2,111,75.225225,0.637033,77.027027,0.637033,-1.801802,1.274066,0
4,christine,distributive,2,814,69.226044,0.086868,68.181818,2.432308,1.044226,2.345440,1
5,christine,thermometer,2,814,67.444717,1.216154,67.567568,1.737363,-0.122850,0.521209,1
6,fashion-mnist,distributive,1,10000,85.680000,NaN,84.310000,NaN,1.370000,NaN,1
7,fashion-mnist,thermometer,1,10000,85.340000,NaN,83.830000,NaN,1.510000,NaN,1
8,higgs,distributive,2,14708,67.908621,0.730760,61.062007,1.413444,6.846614,0.682684,2
9,higgs,thermometer,2,14708,62.251836,0.192305,58.305004,1.466328,3.946832,1.658634,2


In [12]:
def br(value, decimals=2, sign=False):
    """number in pt-BR (decimal comma, thousands dot); the minus sign is U+2212"""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "—"
    text = ("{:+,.%df}" if sign else "{:,.%df}") % decimals
    text = text.format(value).replace(",", "_").replace(".", ",").replace("_", ".")
    if sign and float(value) == 0:
        text = text.lstrip("+-")
    return text.replace("-", "−")


def mean_std(mean, std, sign=False):
    return br(mean, sign=sign) if std is None or math.isnan(std) else "%s ± %s" % (br(mean, sign=sign), br(std))


def count(n, singular, plural):
    return "%d %s" % (n, singular if n == 1 else plural)


def tests(deltas):
    """wins/ties/losses, mean and median, Wilcoxon (non-zero differences) and sign test of a list of Δ (pp)"""
    d = np.asarray(deltas, dtype=float)
    w, t, l = int((d > 0).sum()), int((d == 0).sum()), int((d < 0).sum())
    nonzero = d[d != 0]
    return dict(wins=w, ties=t, losses=l, mean=d.mean(), median=np.median(d),
                wilcoxon=stats.wilcoxon(nonzero).pvalue if len(nonzero) > 1 else float("nan"),
                sign=stats.binomtest(w, w + l).pvalue if w + l > 0 else float("nan"))


def outcome(t):
    return "%s, %s e %s" % (count(t["wins"], "vitória", "vitórias"), count(t["ties"], "empate", "empates"),
                            count(t["losses"], "derrota", "derrotas"))


def markdown_table(encoding):
    rows = long[long["encoding"] == encoding]
    agg = aggregate[aggregate["encoding"] == encoding].set_index("dataset")
    seeds = sorted(rows["seed"].unique())
    header = ["Conjunto", "n teste", "sementes"]
    for s in seeds:
        header += ["Épocas s%d" % s, "One-shot s%d" % s, "Δ s%d" % s, "z s%d" % s]
    header += ["Épocas (média ± dp)", "One-shot (média ± dp)", "Δ (média ± dp)", "Δ > 0"]
    lines = ["### %s — sementes %s" % (ENCODINGS[encoding], ", ".join(map(str, seeds))), "",
             "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name, a in agg.iterrows():
        by_seed = rows[rows["dataset"] == name].set_index("seed")
        cells = [name, br(a["n_test"], 0), str(a["seeds"])]
        for s in seeds:
            if s not in by_seed.index:
                cells += ["—"] * 4
                continue
            r = by_seed.loc[s]
            bold = "**" if abs(r["z"]) >= 1.96 else ""
            cells += [br(r["epochs"]), br(r["oneshot"]), bold + br(r["delta"], sign=True) + bold, bold + br(r["z"]) + bold]
        cells += [mean_std(a["epochs_mean"], a["epochs_std"]), mean_std(a["oneshot_mean"], a["oneshot_std"]),
                  mean_std(a["delta_mean"], a["delta_std"], sign=True), "%d/%d" % (a["wins"], a["seeds"])]
        lines.append("| " + " | ".join(cells) + " |")

    t = tests(agg["delta_mean"])
    lines += ["", "**Resumo — Δ médio de cada conjunto (%d conjuntos):**" % len(agg), "",
              "- %s das épocas;" % outcome(t),
              "- Δ médio %s pp, Δ mediano %s pp;" % (br(t["mean"], sign=True), br(t["median"], sign=True)),
              "- Wilcoxon p = %s e teste de sinais p = %s." % (br(t["wilcoxon"], 3), br(t["sign"], 3)),
              "", "**Por semente:**", ""]
    for s in seeds:
        ts = tests(rows.loc[rows["seed"] == s, "delta"])
        lines.append("- semente %d (%d conjuntos): %s; Δ médio %s pp; Wilcoxon p = %s." % (
            s, ts["wins"] + ts["ties"] + ts["losses"], outcome(ts), br(ts["mean"], sign=True), br(ts["wilcoxon"], 3)))
    return "\n".join(lines)


def detail_table(encoding):
    """validation, oracle and chosen configuration of each seed"""
    rows = long[long["encoding"] == encoding].sort_values(["dataset", "seed"])
    return pd.DataFrame({
        "conjunto": rows["dataset"], "semente": rows["seed"],
        "épocas — val": rows["epochs_val"], "one-shot — val": rows["oneshot_val"],
        "oráculo épocas": rows["oracle_epochs"], "oráculo one-shot": rows["oracle_oneshot"],
        "épocas: bits / n-tupla": rows["epochs_bits"].astype(str) + " / " + rows["epochs_address"].astype(str),
        "épocas: b / ρ₀ / τ": rows["epochs_b"].astype(str) + " / " + rows["epochs_rho0"].astype(str) + " / " + rows["epochs_tau"].astype(str),
        "melhor época / rodadas": rows["best_epoch"].astype(str) + " / " + rows["epochs_run"].astype(str),
        "one-shot: bits / n-tupla": rows["oneshot_bits"].astype(str) + " / " + rows["oneshot_address"].astype(str),
    }).style.format(precision=2, decimal=",").hide(axis="index")

## Termômetro distributivo

In [13]:
display(Markdown(markdown_table("distributive")))

### Termômetro distributivo — sementes 0, 1

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 2 | 83,65 | 79,81 | +3,85 | 0,72 | 86,54 | 81,73 | +4,81 | 0,95 | 85,10 ± 2,04 | 80,77 ± 1,36 | +4,33 ± 0,68 | 2/2 |
| blood | 111 | 2 | 72,07 | 71,17 | +0,90 | 0,15 | 71,17 | 73,87 | −2,70 | −0,45 | 71,62 ± 0,64 | 72,52 ± 1,91 | −0,90 ± 2,55 | 1/2 |
| christine | 814 | 2 | 69,29 | 69,90 | −0,61 | −0,27 | 69,16 | 66,46 | +2,70 | 1,17 | 69,23 ± 0,09 | 68,18 ± 2,43 | +1,04 ± 2,35 | 1/2 |
| fashion-mnist | 10.000 | 1 | — | — | — | — | 85,68 | 84,31 | **+1,37** | **2,71** | 85,68 | 84,31 | +1,37 | 1/1 |
| higgs | 14.708 | 2 | 67,39 | 60,06 | **+7,33** | **13,11** | 68,43 | 62,06 | **+6,36** | **11,49** | 67,91 ± 0,73 | 61,06 ± 1,41 | +6,85 ± 0,68 | 2/2 |
| iris | 21 | 2 | 100,00 | 100,00 | 0,00 | 0,00 | 80,95 | 76,19 | +4,76 | 0,38 | 90,48 ± 13,47 | 88,10 ± 16,84 | +2,38 ± 3,37 | 1/2 |
| jasmine | 448 | 2 | 77,01 | 78,79 | −1,79 | −0,64 | 77,90 | 76,34 | +1,56 | 0,56 | 77,46 ± 0,63 | 77,57 ± 1,74 | −0,11 ± 2,37 | 1/2 |
| jsc | 124.501 | 1 | — | — | — | — | 72,01 | 71,45 | **+0,56** | **3,10** | 72,01 | 71,45 | +0,56 | 1/1 |
| miniboone | 19.510 | 2 | 91,89 | 91,30 | **+0,59** | **2,10** | 91,50 | 90,90 | **+0,59** | **2,07** | 91,69 ± 0,28 | 91,10 ± 0,28 | +0,59 ± 0,00 | 2/2 |
| mnist | 10.000 | 1 | — | — | — | — | 96,11 | 95,62 | +0,49 | 1,74 | 96,11 | 95,62 | +0,49 | 1/1 |
| nomao | 5.169 | 2 | 95,03 | 94,37 | +0,66 | 1,49 | 95,26 | 94,89 | +0,37 | 0,86 | 95,14 ± 0,16 | 94,63 ± 0,37 | +0,51 ± 0,21 | 2/2 |
| phoneme | 810 | 2 | 86,42 | 85,68 | +0,74 | 0,43 | 88,52 | 88,15 | +0,37 | 0,23 | 87,47 ± 1,48 | 86,91 ± 1,75 | +0,56 ± 0,26 | 2/2 |
| segment | 346 | 2 | 99,42 | 99,42 | 0,00 | 0,00 | 98,55 | 98,55 | 0,00 | 0,00 | 98,99 ± 0,61 | 98,99 ± 0,61 | 0,00 ± 0,00 | 0/2 |
| skin-seg | 36.758 | 2 | 99,28 | 99,26 | +0,03 | 0,43 | 99,87 | 99,82 | +0,04 | 1,50 | 99,58 ± 0,41 | 99,54 ± 0,40 | +0,04 ± 0,01 | 2/2 |
| sylvine | 770 | 2 | 92,73 | 87,79 | **+4,94** | **3,28** | 92,60 | 87,92 | **+4,68** | **3,10** | 92,66 ± 0,09 | 87,86 ± 0,09 | +4,81 ± 0,18 | 2/2 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 12 vitórias, 1 empate e 2 derrotas das épocas;
- Δ médio +1,50 pp, Δ mediano +0,56 pp;
- Wilcoxon p = 0,005 e teste de sinais p = 0,013.

**Por semente:**

- semente 0 (12 conjuntos): 8 vitórias, 2 empates e 2 derrotas; Δ médio +1,39 pp; Wilcoxon p = 0,084.
- semente 1 (15 conjuntos): 13 vitórias, 1 empate e 1 derrota; Δ médio +1,73 pp; Wilcoxon p = 0,007.

In [14]:
detail_table("distributive")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"88,35","86,41","86,54","86,54",4 / 4,1 / 0.8 / 0.05,2 / 6,8 / 12
australian,1,"87,38","85,44","90,38","86,54",2 / 2,1 / 0.9 / 0.05,6 / 10,16 / 8
blood,0,"82,30","77,88","80,18","78,38",4 / 8,2 / 0.9 / 0.05,6 / 10,16 / 16
blood,1,"87,61","84,07","79,28","77,48",16 / 16,2 / 0.7 / 0.05,7 / 11,16 / 16
christine,0,"72,78","70,81","72,11","69,90",16 / 16,1 / 0.9 / 0.0,0 / 4,8 / 12
christine,1,"71,92","69,33","70,02","67,08",2 / 4,2 / 0.7 / 0.05,8 / 12,32 / 16
fashion-mnist,1,"86,92","85,21","85,70","84,31",16 / 32,1 / 0.9 / 0.0,0 / 4,16 / 32
higgs,0,"68,28","60,62","68,14","60,06",16 / 8,2 / 0.7 / 0.4,18 / 20,16 / 24
higgs,1,"69,64","61,79","68,43","62,06",32 / 8,1 / 0.9 / 0.1,19 / 20,32 / 24
iris,0,"100,00","100,00","100,00","100,00",4 / 4,1 / 0.1 / 0.0,0 / 4,4 / 4


## Termômetro linear

In [15]:
display(Markdown(markdown_table("thermometer")))

### Termômetro linear — sementes 0, 1

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 2 | 79,81 | 83,65 | −3,85 | −0,72 | 80,77 | 79,81 | +0,96 | 0,17 | 80,29 ± 0,68 | 81,73 ± 2,72 | −1,44 ± 3,40 | 1/2 |
| blood | 111 | 2 | 74,77 | 77,48 | −2,70 | −0,47 | 75,68 | 76,58 | −0,90 | −0,16 | 75,23 ± 0,64 | 77,03 ± 0,64 | −1,80 ± 1,27 | 0/2 |
| christine | 814 | 2 | 68,30 | 68,80 | −0,49 | −0,21 | 66,58 | 66,34 | +0,25 | 0,10 | 67,44 ± 1,22 | 67,57 ± 1,74 | −0,12 ± 0,52 | 1/2 |
| fashion-mnist | 10.000 | 1 | — | — | — | — | 85,34 | 83,83 | **+1,51** | **2,96** | 85,34 | 83,83 | +1,51 | 1/1 |
| higgs | 14.708 | 2 | 62,12 | 59,34 | **+2,77** | **4,87** | 62,39 | 57,27 | **+5,12** | **8,97** | 62,25 ± 0,19 | 58,31 ± 1,47 | +3,95 ± 1,66 | 2/2 |
| iris | 21 | 2 | 95,24 | 95,24 | 0,00 | 0,00 | 95,24 | 95,24 | 0,00 | 0,00 | 95,24 ± 0,00 | 95,24 ± 0,00 | 0,00 ± 0,00 | 0/2 |
| jasmine | 448 | 2 | 73,88 | 73,44 | +0,45 | 0,15 | 78,35 | 75,00 | +3,35 | 1,19 | 76,12 ± 3,16 | 74,22 ± 1,10 | +1,90 ± 2,05 | 2/2 |
| jsc | 124.501 | 1 | — | — | — | — | 70,95 | 68,16 | **+2,80** | **15,17** | 70,95 | 68,16 | +2,80 | 1/1 |
| miniboone | 19.510 | 2 | 85,07 | 83,32 | **+1,76** | **4,76** | 86,59 | 83,20 | **+3,39** | **9,36** | 85,83 ± 1,07 | 83,26 ± 0,08 | +2,57 ± 1,15 | 2/2 |
| mnist | 10.000 | 1 | — | — | — | — | 96,30 | 96,02 | +0,28 | 1,03 | 96,30 | 96,02 | +0,28 | 1/1 |
| nomao | 5.169 | 2 | 94,83 | 94,47 | +0,37 | 0,83 | 95,63 | 95,28 | +0,35 | 0,85 | 95,23 ± 0,56 | 94,87 ± 0,57 | +0,36 ± 0,01 | 2/2 |
| phoneme | 810 | 2 | 83,09 | 81,98 | +1,11 | 0,59 | 85,80 | 85,19 | +0,62 | 0,35 | 84,44 ± 1,92 | 83,58 ± 2,27 | +0,86 ± 0,35 | 2/2 |
| segment | 346 | 2 | 98,84 | 98,84 | 0,00 | 0,00 | 98,27 | 98,27 | 0,00 | 0,00 | 98,55 ± 0,41 | 98,55 ± 0,41 | 0,00 ± 0,00 | 0/2 |
| skin-seg | 36.758 | 2 | 99,62 | 99,50 | **+0,13** | **2,56** | 99,79 | 99,73 | +0,06 | 1,58 | 99,70 ± 0,12 | 99,61 ± 0,17 | +0,09 ± 0,05 | 2/2 |
| sylvine | 770 | 2 | 90,00 | 80,91 | **+9,09** | **5,10** | 90,91 | 87,40 | **+3,51** | **2,22** | 90,45 ± 0,64 | 84,16 ± 4,59 | +6,30 ± 3,95 | 2/2 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 10 vitórias, 2 empates e 3 derrotas das épocas;
- Δ médio +1,15 pp, Δ mediano +0,36 pp;
- Wilcoxon p = 0,040 e teste de sinais p = 0,092.

**Por semente:**

- semente 0 (12 conjuntos): 7 vitórias, 2 empates e 3 derrotas; Δ médio +0,72 pp; Wilcoxon p = 0,492.
- semente 1 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +1,42 pp; Wilcoxon p = 0,003.

In [16]:
detail_table("thermometer")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"86,41","82,52","90,38","88,46",4 / 4,2 / 0.7 / 0.3,2 / 6,4 / 16
australian,1,"88,35","84,47","92,31","89,42",16 / 16,1 / 0.7 / 0.0,0 / 4,16 / 16
blood,0,"84,96","81,42","81,08","78,38",16 / 12,2 / 0.7 / 0.5,5 / 9,16 / 20
blood,1,"84,07","81,42","80,18","80,18",32 / 4,2 / 0.3 / 0.2,0 / 4,8 / 16
christine,0,"71,55","70,57","70,39","69,90",16 / 24,1 / 0.7 / 0.1,0 / 4,16 / 24
christine,1,"71,43","70,94","68,06","66,83",32 / 24,1 / 0.5 / 0.0,0 / 4,32 / 24
fashion-mnist,1,"86,66","84,67","85,56","83,93",16 / 32,1 / 0.9 / 0.0,0 / 4,16 / 40
higgs,0,"62,45","58,94","62,16","59,34",16 / 4,1 / 0.1 / 0.5,7 / 11,8 / 12
higgs,1,"63,49","57,49","62,39","57,27",32 / 2,2 / 0.1 / 0.05,3 / 7,32 / 4
iris,0,"100,00","100,00","100,00","100,00",4 / 4,1 / 0.1 / 0.0,0 / 4,4 / 4


## DataFrames para exportar

- `long`: uma linha por conjunto, binarização e semente (todas as colunas);
- `aggregate`: uma linha por conjunto e binarização, com média e desvio entre as sementes.

Por exemplo: `long.to_csv("benchmark_por_semente.csv", index=False, decimal=",", sep=";")`.